# Standalone model and human-assisted accuracy

Compare manual web-interface and API model benchmarks with post-consultation participant performance.

## Load and summarize the manual benchmark

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.stats.proportion import proportion_confint

ROOT = Path("..").resolve()
MANUAL_PATH = ROOT / "data" / "Psy_Case_Vignette_Data_Manual_Copy_Paste.xlsx"
PROLIFIC_PATH = ROOT / "data" / "Psy_Case_Vignette_Data_Prolific_analysis.xlsx"
OUT_DIR = ROOT / "output"
FIG_DIR = ROOT / "manuscript" / "Figures"

VIGNETTES = ["A", "B", "C", "D", "E"]
CLINICAL_VIGNETTES = ["A", "B", "C", "D"]
CASE_TO_VIGNETTE = {1: "A", 2: "B", 3: "C", 4: "D", 5: "E"}
VIGNETTE_LABELS = {
    "A": "A (Psychosis)",
    "B": "B (Panic Attack)",
    "C": "C (OCD)",
    "D": "D (Bipolar / Mania)",
    "E": "E (Control case)",
}

PROVIDER_ORDER = ["ChatGPT", "Copilot", "Perplexity"]
ARM_TO_PROVIDER = {1: "ChatGPT", 2: "Copilot", 3: "Perplexity"}

OUTCOMES = [
    {
        "outcome": "Diagnosis",
        "manual_col": "Diagnosis_correct",
        "prolific_suffix": "post_condition_correct",
    },
    {
        "outcome": "Triage (next step)",
        "manual_col": "Next_Step_Correct",
        "prolific_suffix": "post_nextstep_correct",
    },
]

manual = pd.read_excel(MANUAL_PATH)
prolific = pd.read_excel(PROLIFIC_PATH)

manual["Vignette"] = manual["Case"].map(CASE_TO_VIGNETTE)
prolific_tool = prolific.loc[prolific["study_arm"].isin(ARM_TO_PROVIDER.keys())].copy()
prolific_tool["Provider"] = prolific_tool["study_arm"].map(ARM_TO_PROVIDER)

print(f"Manual benchmark: {len(manual):,} rows")
print(f"Prolific tool arms: {len(prolific_tool):,} participants")
print("Providers:", manual["Provider"].unique().tolist())

### Calculate benchmark accuracy

In [ ]:
def wilson_ci(n_correct: int, n_total: int, alpha: float = 0.05) -> tuple[float, float]:
    if n_total == 0:
        return float("nan"), float("nan")
    lo, hi = proportion_confint(n_correct, n_total, alpha=alpha, method="wilson")
    return lo * 100, hi * 100


def summarize_binary(series: pd.Series) -> dict:
    valid = series.dropna()
    n_total = len(valid)
    n_correct = int((valid == 1).sum())
    pct = n_correct / n_total * 100 if n_total else float("nan")
    ci_lo, ci_hi = wilson_ci(n_correct, n_total)
    return {
        "n_correct": n_correct,
        "n_total": n_total,
        "pct": round(pct, 1) if n_total else float("nan"),
        "ci_lo": round(ci_lo, 1) if n_total else float("nan"),
        "ci_hi": round(ci_hi, 1) if n_total else float("nan"),
    }


def fmt_n_pct(stats: dict) -> str:
    if stats["n_total"] == 0:
        return ""
    return f"{stats['n_correct']}/{stats['n_total']} ({stats['pct']:.1f}%)"


def fmt_ci(stats: dict) -> str:
    if stats["n_total"] == 0 or pd.isna(stats["ci_lo"]):
        return ""
    return f"[{stats['ci_lo']:.1f}, {stats['ci_hi']:.1f}]"


def manual_subset(provider: str, vignette: str, manual_col: str) -> pd.Series:
    return manual.loc[
        (manual["Provider"] == provider) & (manual["Vignette"] == vignette),
        manual_col,
    ]


def prolific_subset(provider: str, vignette: str, prolific_suffix: str) -> pd.Series:
    col = f"vig_{vignette}_{prolific_suffix}"
    return prolific_tool.loc[prolific_tool["Provider"] == provider, col]


def build_comparison_table(outcome_cfg: dict) -> pd.DataFrame:
    rows = []
    manual_col = outcome_cfg["manual_col"]
    prolific_suffix = outcome_cfg["prolific_suffix"]

    for provider in PROVIDER_ORDER:
        vignette_rows = []
        for vignette in VIGNETTES:
            model = summarize_binary(manual_subset(provider, vignette, manual_col))
            human = summarize_binary(prolific_subset(provider, vignette, prolific_suffix))
            delta = model["pct"] - human["pct"] if model["n_total"] and human["n_total"] else float("nan")
            row = {
                "Provider": provider,
                "Vignette": vignette,
                "Vignette label": VIGNETTE_LABELS[vignette],
                "Model n/N (pct)": fmt_n_pct(model),
                "Model 95% CI": fmt_ci(model),
                "Human post n/N (pct)": fmt_n_pct(human),
                "Human post 95% CI": fmt_ci(human),
                "Delta pp (model - human)": round(delta, 1) if pd.notna(delta) else "",
            }
            rows.append(row)
            vignette_rows.append((model, human))

        model_clinical = summarize_binary(
            pd.concat(
                [manual_subset(provider, v, manual_col) for v in CLINICAL_VIGNETTES],
                ignore_index=True,
            )
        )
        human_clinical = summarize_binary(
            pd.concat(
                [prolific_subset(provider, v, prolific_suffix) for v in CLINICAL_VIGNETTES],
                ignore_index=True,
            )
        )
        delta_clinical = model_clinical["pct"] - human_clinical["pct"]
        rows.append(
            {
                "Provider": provider,
                "Vignette": "A-D",
                "Vignette label": "Clinical vignettes (A\u2013D, pooled)",
                "Model n/N (pct)": fmt_n_pct(model_clinical),
                "Model 95% CI": fmt_ci(model_clinical),
                "Human post n/N (pct)": fmt_n_pct(human_clinical),
                "Human post 95% CI": fmt_ci(human_clinical),
                "Delta pp (model - human)": round(delta_clinical, 1),
            }
        )

        model_pool = summarize_binary(
            pd.concat(
                [manual_subset(provider, v, manual_col) for v in VIGNETTES],
                ignore_index=True,
            )
        )
        human_pool = summarize_binary(
            pd.concat(
                [prolific_subset(provider, v, prolific_suffix) for v in VIGNETTES],
                ignore_index=True,
            )
        )
        delta_pool = model_pool["pct"] - human_pool["pct"]
        rows.append(
            {
                "Provider": provider,
                "Vignette": "All",
                "Vignette label": "All vignettes (pooled)",
                "Model n/N (pct)": fmt_n_pct(model_pool),
                "Model 95% CI": fmt_ci(model_pool),
                "Human post n/N (pct)": fmt_n_pct(human_pool),
                "Human post 95% CI": fmt_ci(human_pool),
                "Delta pp (model - human)": round(delta_pool, 1),
            }
        )

    return pd.DataFrame(rows)


comparison_tables = {}
for cfg in OUTCOMES:
    comparison_tables[cfg["outcome"]] = build_comparison_table(cfg)

for outcome, table in comparison_tables.items():
    print("=" * 80)
    print(outcome)
    print("=" * 80)
    print(table.to_string(index=False))
    print()

## Compare the manual benchmark with participant performance

Use independent Fisher exact or two-proportion tests for provider-matched accuracy counts.

In [ ]:
from scipy.stats import fisher_exact
from statsmodels.stats.proportion import proportions_ztest


def compare_accuracy(model_stats: dict, human_stats: dict) -> dict:
    """Two-sample test: model benchmark vs Prolific post accuracy."""
    a, n1 = model_stats["n_correct"], model_stats["n_total"]
    b, n2 = human_stats["n_correct"], human_stats["n_total"]
    if n1 == 0 or n2 == 0:
        return {
            "test": "",
            "p_two_sided": float("nan"),
            "p_model_better": float("nan"),
        }

    use_fisher = min(n1, n2) < 20
    table = [[a, n1 - a], [b, n2 - b]]

    if use_fisher:
        _, p_two = fisher_exact(table, alternative="two-sided")
        _, p_greater = fisher_exact(table, alternative="greater")
        test_name = "Fisher"
    else:
        _, p_two = proportions_ztest([a, b], [n1, n2], alternative="two-sided")
        _, p_greater = proportions_ztest([a, b], [n1, n2], alternative="larger")
        test_name = "Z-test"

    return {
        "test": test_name,
        "p_two_sided": p_two,
        "p_model_better": p_greater,
    }


TEST_POOLS = [
    ("A-D", CLINICAL_VIGNETTES),
    ("All", VIGNETTES),
]


def build_test_table(
    benchmark: pd.DataFrame,
    prolific_df: pd.DataFrame,
    provider_order: list[str],
    outcome_cfg: dict,
    *,
    benchmark_subset_fn,
    prolific_subset_fn,
) -> pd.DataFrame:
    benchmark_col = outcome_cfg.get("manual_col") or outcome_cfg["benchmark_col"]
    prolific_suffix = outcome_cfg["prolific_suffix"]
    rows = []

    for provider in provider_order:
        for vignette in VIGNETTES:
            model_series = benchmark_subset_fn(benchmark, provider, vignette, benchmark_col)
            human_series = prolific_subset_fn(prolific_df, provider, vignette, prolific_suffix)
            model_stats = summarize_binary(model_series)
            human_stats = summarize_binary(human_series)
            test = compare_accuracy(model_stats, human_stats)
            rows.append(
                {
                    "Provider": provider,
                    "Vignette": vignette,
                    "Model n/N (pct)": fmt_n_pct(model_stats),
                    "Human post n/N (pct)": fmt_n_pct(human_stats),
                    "Delta pp (model - human)": round(model_stats["pct"] - human_stats["pct"], 1)
                    if model_stats["n_total"] and human_stats["n_total"]
                    else "",
                    "Test": test["test"],
                    "p (two-sided)": round(test["p_two_sided"], 4) if pd.notna(test["p_two_sided"]) else "",
                    "p (model > human)": round(test["p_model_better"], 4)
                    if pd.notna(test["p_model_better"])
                    else "",
                }
            )

        for pool_label, pool_vignettes in TEST_POOLS:
            model_series = pd.concat(
                [benchmark_subset_fn(benchmark, provider, v, benchmark_col) for v in pool_vignettes],
                ignore_index=True,
            )
            human_series = pd.concat(
                [prolific_subset_fn(prolific_df, provider, v, prolific_suffix) for v in pool_vignettes],
                ignore_index=True,
            )
            model_stats = summarize_binary(model_series)
            human_stats = summarize_binary(human_series)
            test = compare_accuracy(model_stats, human_stats)
            rows.append(
                {
                    "Provider": provider,
                    "Vignette": pool_label,
                    "Model n/N (pct)": fmt_n_pct(model_stats),
                    "Human post n/N (pct)": fmt_n_pct(human_stats),
                    "Delta pp (model - human)": round(model_stats["pct"] - human_stats["pct"], 1)
                    if model_stats["n_total"] and human_stats["n_total"]
                    else "",
                    "Test": test["test"],
                    "p (two-sided)": round(test["p_two_sided"], 4) if pd.notna(test["p_two_sided"]) else "",
                    "p (model > human)": round(test["p_model_better"], 4)
                    if pd.notna(test["p_model_better"])
                    else "",
                }
            )

    return pd.DataFrame(rows)


def benchmark_subset_manual(benchmark, provider, vignette, col):
    return benchmark.loc[(benchmark["Provider"] == provider) & (benchmark["Vignette"] == vignette), col]


def prolific_subset_tool(prolific_df, provider, vignette, suffix):
    col = f"vig_{vignette}_{suffix}"
    return prolific_df.loc[prolific_df["Provider"] == provider, col]


manual_test_tables = {}
for cfg in OUTCOMES:
    manual_test_tables[cfg["outcome"]] = build_test_table(
        manual,
        prolific_tool,
        PROVIDER_ORDER,
        cfg,
        benchmark_subset_fn=benchmark_subset_manual,
        prolific_subset_fn=prolific_subset_tool,
    )

for outcome, table in manual_test_tables.items():
    print("=" * 80)
    print(f"Manual copy-paste — {outcome}")
    print("=" * 80)
    print(table.to_string(index=False))
    print()

## Load and summarize the API benchmark

In [ ]:
API_PATH = ROOT / "data" / "Psy_Case_Vignette_Data_API.xlsx"

API_CASE_TO_VIGNETTE = {
    "case_1": "A",
    "case_2": "B",
    "case_3": "C",
    "case_4": "D",
    "case_5": "E",
}
API_PROVIDER_MAP = {"openai": "ChatGPT", "perplexity": "Perplexity"}
API_PROVIDER_ORDER = ["ChatGPT", "Perplexity"]
API_ARM_TO_PROVIDER = {1: "ChatGPT", 3: "Perplexity"}

API_OUTCOMES = [
    {
        "outcome": "Diagnosis",
        "benchmark_col": "diagnosis_correct",
        "prolific_suffix": "post_condition_correct",
    },
    {
        "outcome": "Triage (next step)",
        "benchmark_col": "next_step_correct_category",
        "prolific_suffix": "post_nextstep_correct",
    },
]

api = pd.read_excel(API_PATH)
api["Vignette"] = api["case_id"].map(API_CASE_TO_VIGNETTE)
api["Provider"] = api["provider"].map(API_PROVIDER_MAP)

prolific_api = prolific.loc[prolific["study_arm"].isin(API_ARM_TO_PROVIDER.keys())].copy()
prolific_api["Provider"] = prolific_api["study_arm"].map(API_ARM_TO_PROVIDER)

print(f"API benchmark: {len(api):,} rows")
print(f"Prolific API arms: {len(prolific_api):,} participants")
print("API providers:", api["Provider"].unique().tolist())


def benchmark_subset(
    benchmark: pd.DataFrame, provider: str, vignette: str, benchmark_col: str
) -> pd.Series:
    return benchmark.loc[
        (benchmark["Provider"] == provider) & (benchmark["Vignette"] == vignette),
        benchmark_col,
    ]


def prolific_subset_df(
    prolific_df: pd.DataFrame, provider: str, vignette: str, prolific_suffix: str
) -> pd.Series:
    col = f"vig_{vignette}_{prolific_suffix}"
    return prolific_df.loc[prolific_df["Provider"] == provider, col]


def build_comparison_table_benchmark(
    benchmark: pd.DataFrame,
    prolific_df: pd.DataFrame,
    provider_order: list[str],
    outcome_cfg: dict,
) -> pd.DataFrame:
    rows = []
    benchmark_col = outcome_cfg["benchmark_col"]
    prolific_suffix = outcome_cfg["prolific_suffix"]

    for provider in provider_order:
        for vignette in VIGNETTES:
            model = summarize_binary(benchmark_subset(benchmark, provider, vignette, benchmark_col))
            human = summarize_binary(prolific_subset_df(prolific_df, provider, vignette, prolific_suffix))
            delta = model["pct"] - human["pct"] if model["n_total"] and human["n_total"] else float("nan")
            rows.append(
                {
                    "Provider": provider,
                    "Vignette": vignette,
                    "Vignette label": VIGNETTE_LABELS[vignette],
                    "Model n/N (pct)": fmt_n_pct(model),
                    "Model 95% CI": fmt_ci(model),
                    "Human post n/N (pct)": fmt_n_pct(human),
                    "Human post 95% CI": fmt_ci(human),
                    "Delta pp (model - human)": round(delta, 1) if pd.notna(delta) else "",
                }
            )

        model_clinical = summarize_binary(
            pd.concat(
                [benchmark_subset(benchmark, provider, v, benchmark_col) for v in CLINICAL_VIGNETTES],
                ignore_index=True,
            )
        )
        human_clinical = summarize_binary(
            pd.concat(
                [prolific_subset_df(prolific_df, provider, v, prolific_suffix) for v in CLINICAL_VIGNETTES],
                ignore_index=True,
            )
        )
        delta_clinical = model_clinical["pct"] - human_clinical["pct"]
        rows.append(
            {
                "Provider": provider,
                "Vignette": "A-D",
                "Vignette label": "Clinical vignettes (A\u2013D, pooled)",
                "Model n/N (pct)": fmt_n_pct(model_clinical),
                "Model 95% CI": fmt_ci(model_clinical),
                "Human post n/N (pct)": fmt_n_pct(human_clinical),
                "Human post 95% CI": fmt_ci(human_clinical),
                "Delta pp (model - human)": round(delta_clinical, 1),
            }
        )

        model_pool = summarize_binary(
            pd.concat(
                [benchmark_subset(benchmark, provider, v, benchmark_col) for v in VIGNETTES],
                ignore_index=True,
            )
        )
        human_pool = summarize_binary(
            pd.concat(
                [prolific_subset_df(prolific_df, provider, v, prolific_suffix) for v in VIGNETTES],
                ignore_index=True,
            )
        )
        delta_pool = model_pool["pct"] - human_pool["pct"]
        rows.append(
            {
                "Provider": provider,
                "Vignette": "All",
                "Vignette label": "All vignettes (pooled)",
                "Model n/N (pct)": fmt_n_pct(model_pool),
                "Model 95% CI": fmt_ci(model_pool),
                "Human post n/N (pct)": fmt_n_pct(human_pool),
                "Human post 95% CI": fmt_ci(human_pool),
                "Delta pp (model - human)": round(delta_pool, 1),
            }
        )

    return pd.DataFrame(rows)


comparison_tables_api = {}
for cfg in API_OUTCOMES:
    comparison_tables_api[cfg["outcome"]] = build_comparison_table_benchmark(
        api, prolific_api, API_PROVIDER_ORDER, cfg
    )

for outcome, table in comparison_tables_api.items():
    print("=" * 80)
    print(f"API — {outcome}")
    print("=" * 80)
    print(table.to_string(index=False))
    print()

excel_api_path = OUT_DIR / "standalone_model_vs_human_post_api.xlsx"
OUT_DIR.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(excel_api_path, engine="openpyxl") as writer:
    for outcome, table in comparison_tables_api.items():
        sheet = outcome.replace(" ", "_")[:31]
        table.to_excel(writer, sheet_name=sheet, index=False)
print(f"Saved {excel_api_path}")

## Compare the API benchmark with participant performance

In [ ]:
api_test_tables = {}
for cfg in API_OUTCOMES:
    api_test_tables[cfg["outcome"]] = build_test_table(
        api,
        prolific_api,
        API_PROVIDER_ORDER,
        cfg,
        benchmark_subset_fn=benchmark_subset,
        prolific_subset_fn=prolific_subset_df,
    )

for outcome, table in api_test_tables.items():
    print("=" * 80)
    print(f"API — {outcome}")
    print("=" * 80)
    print(table.to_string(index=False))
    print()

## Plot standalone and human-assisted accuracy

In [ ]:
import matplotlib.lines as mlines

FIG_DIR.mkdir(parents=True, exist_ok=True)

CONTROL_VIGNETTE = "E"
PANEL_COLUMNS = [
    ("Clinical vignettes (A\u2013D, pooled)", CLINICAL_VIGNETTES),
    ("Control vignette (E)", [CONTROL_VIGNETTE]),
]
PANEL_LETTERS = ["A", "B", "C", "D"]
OUTCOME_DISPLAY = {
    "Diagnosis": "Condition diagnosis",
    "Triage (next step)": "Triage (next step)",
}
SOURCE_ORDER = ["Human post-tool", "Manual copy-paste", "API"]
PROVIDER_COLORS = {
    "ChatGPT": "#009E73",
    "Copilot": "#0072B2",
    "Perplexity": "#E69F00",
}
HUMAN_COLOR = "#999999"
CI_COLOR = "#888888"
SOURCE_OFFSETS = {
    "Human post-tool": -0.14,
    "Manual copy-paste": 0.0,
    "API": 0.14,
}
STAR_Y_OFFSET = {
    "Manual copy-paste": 1.8,
    "API": 3.6,
}
MARKER_SIZE = 6.5
MARKER_EDGEWIDTH = 1.4
API_OUTCOME_COL = {
    "Diagnosis": "diagnosis_correct",
    "Triage (next step)": "next_step_correct_category",
}


def p_to_stars(p: float) -> str:
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""


def build_panel_sig_pvalues(outcome_cfg: dict, vignettes: list[str]) -> dict[tuple[str, str], float]:
    """Two-sided p-values for manual/API vs human post-tool at the panel pooling level."""
    prolific_suffix = outcome_cfg["prolific_suffix"]
    manual_col = outcome_cfg["manual_col"]
    api_col = API_OUTCOME_COL[outcome_cfg["outcome"]]
    pvalues = {}

    for provider in PROVIDER_ORDER:
        human_stats = summarize_binary(
            pooled_human_series(provider, vignettes, prolific_suffix)
        )
        manual_stats = summarize_binary(
            pooled_manual_series(provider, vignettes, manual_col)
        )
        pvalues[(provider, "Manual copy-paste")] = compare_accuracy(manual_stats, human_stats)["p_two_sided"]

        if provider in API_PROVIDER_ORDER:
            api_stats = summarize_binary(
                pooled_api_series(provider, vignettes, api_col)
            )
            pvalues[(provider, "API")] = compare_accuracy(api_stats, human_stats)["p_two_sided"]

    return pvalues


def pooled_human_series(provider: str, vignettes: list[str], prolific_suffix: str) -> pd.Series:
    return pd.concat(
        [prolific_subset_tool(prolific_tool, provider, v, prolific_suffix) for v in vignettes],
        ignore_index=True,
    )


def pooled_manual_series(provider: str, vignettes: list[str], manual_col: str) -> pd.Series:
    return pd.concat(
        [manual_subset(provider, v, manual_col) for v in vignettes],
        ignore_index=True,
    )


def pooled_api_series(provider: str, vignettes: list[str], api_col: str) -> pd.Series:
    return pd.concat(
        [benchmark_subset(api, provider, v, api_col) for v in vignettes],
        ignore_index=True,
    )


def build_pooled_panel(outcome_cfg: dict, vignettes: list[str]) -> pd.DataFrame:
    outcome = outcome_cfg["outcome"]
    rows = []
    for provider in PROVIDER_ORDER:
        human = summarize_binary(
            pooled_human_series(provider, vignettes, outcome_cfg["prolific_suffix"])
        )
        manual = summarize_binary(
            pooled_manual_series(provider, vignettes, outcome_cfg["manual_col"])
        )
        rows.append({"provider": provider, "source": "Human post-tool", **human})
        rows.append({"provider": provider, "source": "Manual copy-paste", **manual})
        if provider in API_PROVIDER_ORDER:
            api_stats = summarize_binary(
                pooled_api_series(provider, vignettes, API_OUTCOME_COL[outcome])
            )
            rows.append({"provider": provider, "source": "API", **api_stats})
    return pd.DataFrame(rows)


def plot_pooled_panel(
    ax,
    panel_df: pd.DataFrame,
    *,
    show_ylabel: bool,
    show_xticklabels: bool,
    title: str,
    panel_letter: str = "",
    sig_pvalues: dict[tuple[str, str], float] | None = None,
) -> tuple[float, float]:
    x = np.arange(len(PROVIDER_ORDER))
    max_top = 0.0
    min_bottom = 100.0

    for i, provider in enumerate(PROVIDER_ORDER):
        provider_df = panel_df[panel_df["provider"] == provider]
        x_center = x[i]

        for source in SOURCE_ORDER:
            if source == "API" and provider not in API_PROVIDER_ORDER:
                continue
            row = provider_df[provider_df["source"] == source]
            if row.empty or row.iloc[0]["n_total"] == 0:
                continue
            row = row.iloc[0]
            xpos = x_center + SOURCE_OFFSETS[source]
            max_top = max(max_top, row["ci_hi"])
            min_bottom = min(min_bottom, row["ci_lo"])

            if source == "Human post-tool":
                color = HUMAN_COLOR
                ax.errorbar(
                    xpos,
                    row["pct"],
                    yerr=[[row["pct"] - row["ci_lo"]], [row["ci_hi"] - row["pct"]]],
                    fmt="o",
                    color=CI_COLOR,
                    markerfacecolor=color,
                    markeredgecolor=color,
                    markersize=MARKER_SIZE,
                    markeredgewidth=MARKER_EDGEWIDTH,
                    elinewidth=1.0,
                    capsize=3,
                    capthick=1.0,
                    zorder=3,
                )
            elif source == "Manual copy-paste":
                color = PROVIDER_COLORS[provider]
                ax.errorbar(
                    xpos,
                    row["pct"],
                    yerr=[[row["pct"] - row["ci_lo"]], [row["ci_hi"] - row["pct"]]],
                    fmt="o",
                    color=CI_COLOR,
                    markerfacecolor=color,
                    markeredgecolor=color,
                    markersize=MARKER_SIZE,
                    markeredgewidth=MARKER_EDGEWIDTH,
                    elinewidth=1.0,
                    capsize=3,
                    capthick=1.0,
                    zorder=3,
                )
            else:
                color = PROVIDER_COLORS[provider]
                ax.errorbar(
                    xpos,
                    row["pct"],
                    yerr=[[row["pct"] - row["ci_lo"]], [row["ci_hi"] - row["pct"]]],
                    fmt="o",
                    color=CI_COLOR,
                    markerfacecolor="white",
                    markeredgecolor=color,
                    markersize=MARKER_SIZE,
                    markeredgewidth=MARKER_EDGEWIDTH,
                    elinewidth=1.0,
                    capsize=3,
                    capthick=1.0,
                    zorder=3,
                )

            if source != "Human post-tool" and sig_pvalues is not None:
                stars = p_to_stars(sig_pvalues.get((provider, source), float("nan")))
                if stars:
                    y_star = row["ci_hi"] + STAR_Y_OFFSET[source]
                    ax.text(
                        xpos,
                        y_star,
                        stars,
                        ha="center",
                        va="bottom",
                        fontsize=9,
                        color="#333333",
                        zorder=5,
                    )
                    max_top = max(max_top, y_star + 2)

    ax.set_xticks(x)
    ax.set_xticklabels(PROVIDER_ORDER if show_xticklabels else [""] * len(PROVIDER_ORDER))
    ax.set_xlim(-0.35, len(PROVIDER_ORDER) - 0.65)
    ax.tick_params(axis="x", pad=6)
    ax.set_ylabel("")
    if title:
        ax.set_title(title, loc="left", fontsize=10.5, fontweight="bold", pad=8)
    if panel_letter and not title:
        ax.text(
            -0.10,
            1.04,
            f"({panel_letter})",
            transform=ax.transAxes,
            ha="left",
            va="bottom",
            fontsize=10.5,
            fontweight="bold",
            clip_on=False,
        )
    ax.yaxis.grid(True, linestyle="-", alpha=0.25, color="#cccccc")
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    return max_top, min_bottom


plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.size": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.linewidth": 0.8,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
    }
)
fig, axes = plt.subplots(2, 2, figsize=(9.5, 7.2), sharey=True)
fig.subplots_adjust(wspace=0.24, hspace=0.58, top=0.86, bottom=0.16, left=0.13, right=0.98)

panel_idx = 0
panel_limits = []
for row_idx, outcome_cfg in enumerate(OUTCOMES):
    outcome_label = OUTCOME_DISPLAY[outcome_cfg["outcome"]]
    for col_idx, (col_title, vignettes) in enumerate(PANEL_COLUMNS):
        ax = axes[row_idx, col_idx]
        panel_df = build_pooled_panel(outcome_cfg, vignettes)
        sig_pvalues = build_panel_sig_pvalues(outcome_cfg, vignettes)
        panel_letter = PANEL_LETTERS[panel_idx]
        panel_title = f"({panel_letter}) {col_title}" if row_idx == 0 else ""
        panel_limits.append(
            plot_pooled_panel(
                ax,
                panel_df,
                show_ylabel=(row_idx == 0 and col_idx == 0),
                show_xticklabels=True,
                title=panel_title,
                panel_letter=panel_letter,
                sig_pvalues=sig_pvalues,
            )
        )
        panel_idx += 1

    row_y = 0.76 if row_idx == 0 else 0.31
    fig.text(
        0.012,
        row_y,
        outcome_label,
        ha="center",
        va="center",
        rotation=90,
        fontsize=11,
        fontweight="bold",
    )

fig.text(
    0.055,
    0.5,
    "Percent correct",
    ha="center",
    va="center",
    rotation=90,
    fontsize=10,
)

y_top = min(108, max(lim[0] for lim in panel_limits) + 6)
y_bottom = max(-4, min(lim[1] for lim in panel_limits) - 4)
for ax in axes.flat:
    ax.set_ylim(y_bottom, y_top)
    ax.set_yticks(np.arange(0, 101, 20))

legend_handles = [
    mlines.Line2D(
        [0], [0], marker="o", color="w", linestyle="None",
        markersize=MARKER_SIZE, markerfacecolor=HUMAN_COLOR,
        markeredgecolor=HUMAN_COLOR, markeredgewidth=MARKER_EDGEWIDTH,
        label="Human post-tool",
    ),
    mlines.Line2D(
        [0], [0], marker="o", color="w", linestyle="None",
        markersize=MARKER_SIZE, markerfacecolor=PROVIDER_COLORS["ChatGPT"],
        markeredgecolor=PROVIDER_COLORS["ChatGPT"], markeredgewidth=MARKER_EDGEWIDTH,
        label="Manual copy-paste",
    ),
    mlines.Line2D(
        [0], [0], marker="o", color="w", linestyle="None",
        markersize=MARKER_SIZE, markerfacecolor="white",
        markeredgecolor=PROVIDER_COLORS["ChatGPT"], markeredgewidth=MARKER_EDGEWIDTH,
        label="API (ChatGPT & Perplexity)",
    ),
]
fig.legend(
    handles=legend_handles,
    loc="upper center",
    ncol=3,
    frameon=False,
    bbox_to_anchor=(0.5, 1.0),
)

fig_png = FIG_DIR / "Figure_standalone_benchmark.png"
fig_pdf = FIG_DIR / "Figure_standalone_benchmark.pdf"
fig.savefig(fig_png, dpi=300, bbox_inches="tight")
fig.savefig(fig_pdf, bbox_inches="tight")
print(f"Saved {fig_png}")
print(f"Saved {fig_pdf}")
plt.show()
